# CLR Training Notebook
Unified training notebook for all model types (`pi0`, `pi05`, `xvla`, `wall_x`).
Set `MODEL_TYPE` below and the notebook will load the correct experiment config and launch training.

In [ ]:
#!pip uninstall -y transformers
#!pip install git+https://github.com/huggingface/transformers.git@fix/lerobot_openpi

In [1]:
# @title Parameters
MODEL_TYPE = "smolvla"  # @param ["pi0", "pi05", "xvla", "wall_x", "smolvla"]
EXPERIMENT_CONFIG = ""  # @param {"type":"string"}
# If EXPERIMENT_CONFIG is empty, defaults to experiment-{MODEL_TYPE}.cfg

In [2]:
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()  # loads HF_TOKEN and WANDB_API_KEY from .env

## Set NCCL environment variables for distributed training in GCP G4
## single instance, multi-GPU setup. Adjust the interface name (e.g., "ens3") as needed.
os.environ.pop("NCCL_NET", None)
os.environ["NCCL_SOCKET_IFNAME"] = "ens3"
os.environ["NCCL_P2P_LEVEL"] = "PHB"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

In [3]:
!rm -rf ckpt  # remove checkpoint directory if it already exists to avoid conflicts with previous runs
!rm -rf logs  # remove logs directory if it already exists to avoid conflicts with previous runs

In [4]:
import configparser

SUPPORTED_MODELS = ["pi0", "pi05", "xvla", "wall_x", "smolvla"]
if MODEL_TYPE not in SUPPORTED_MODELS:
    raise ValueError(f"Unknown MODEL_TYPE '{MODEL_TYPE}'. Choose from: {SUPPORTED_MODELS}")

config = configparser.ConfigParser()
config_file = EXPERIMENT_CONFIG if EXPERIMENT_CONFIG else f"experiment-{MODEL_TYPE}.cfg"
config.read(config_file)
print(f"Loaded config: {config_file}")
exp = config["experiment"]

DATASET_ROOT   = exp["DATASET_ROOT"]
DATASET_REPO   = exp["DATASET_REPO"]
POLICY_REPO    = exp["POLICY_REPO"]
OUTPUT_DIR     = exp["OUTPUT_DIR"]
JOB_NAME       = exp["JOB_NAME"] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MAX_TRAIN_STEPS = int(exp["MAX_TRAIN_STEPS"])
CHUNK_SIZE     = int(exp["CHUNK_SIZE"])
ACTION_STEPS   = int(exp["ACTION_STEPS"])
BATCH_SIZE     = int(exp["BATCH_SIZE"])

print(f"MODEL_TYPE:      {MODEL_TYPE}")
print(f"DATASET_ROOT:    {DATASET_ROOT}")
print(f"DATASET_REPO:    {DATASET_REPO}")
print(f"POLICY_REPO:     {POLICY_REPO}")
print(f"OUTPUT_DIR:      {OUTPUT_DIR}")
print(f"JOB_NAME:        {JOB_NAME}")
print(f"MAX_TRAIN_STEPS: {MAX_TRAIN_STEPS}")
print(f"CHUNK_SIZE:      {CHUNK_SIZE}")
print(f"ACTION_STEPS:    {ACTION_STEPS}")
print(f"BATCH_SIZE:      {BATCH_SIZE}")

Loaded config: experiment-smolvla.cfg
MODEL_TYPE:      smolvla
DATASET_ROOT:    ./dataset/teleoperation_clr_dataset
DATASET_REPO:    gimarchetti/clr-experiment-dataset
POLICY_REPO:     gimarchetti/clr-experiment-smolvla
OUTPUT_DIR:      ./ckpt/clr-experiment-smolvla
JOB_NAME:        clr-experiment-smolvla2026-04-13_22-10-28
MAX_TRAIN_STEPS: 1000
CHUNK_SIZE:      50
ACTION_STEPS:    10
BATCH_SIZE:      32


In [5]:
# Download dataset from Hugging Face Hub
!hf download {DATASET_REPO} --repo-type dataset --local-dir {DATASET_ROOT}

Fetching 174 files: 100%|███████████████████| 174/174 [00:00<00:00, 2429.44it/s]
/mnt/content/clr_ws/src/vla/Lerobot-Mujoco/dataset/teleoperation_clr_dataset


In [ ]:
# Build model-specific training flags
if MODEL_TYPE == "pi0":
    MODEL_FLAGS = f"""\
    --policy.type=pi0 \
    --policy.pretrained_path=lerobot/pi0_base \
    --policy.compile_model=false \
    --policy.gradient_checkpointing=true \
    --policy.freeze_vision_encoder=false \
    --policy.train_expert_only=false \
    --policy.dtype=bfloat16 \
    --policy.device=cuda"""

elif MODEL_TYPE == "pi05":
    MODEL_FLAGS = f"""\
    --policy.type=pi05 \
    --policy.pretrained_path=lerobot/pi05_base \
    --policy.compile_model=false \
    --policy.gradient_checkpointing=true \
    --policy.freeze_vision_encoder=false \
    --policy.train_expert_only=false \
    --policy.dtype=bfloat16 \
    --policy.device=cuda"""

elif MODEL_TYPE == "xvla":
    MODEL_FLAGS = f"""\
    --policy.path=lerobot/xvla-base \
    --policy.freeze_vision_encoder=false \
    --policy.freeze_language_encoder=true \
    --policy.train_policy_transformer=true \
    --policy.train_soft_prompts=true \
    --policy.action_mode=auto \
    --policy.dtype=bfloat16 \
    --policy.device=cuda \
    --rename_map='{{"observation.image": "observation.images.image", "observation.left_scene_image": "observation.images.image2", "observation.wrist_image": "observation.images.image3"}}'"""

elif MODEL_TYPE == "wall_x":
    MODEL_FLAGS = f"""\
    --policy.type=wall_x \
    --policy.pretrained_name_or_path=x-square-robot/wall-oss-flow \
    --policy.prediction_mode=diffusion \
    --policy.attn_implementation=eager \
    --policy.device=cuda"""

elif MODEL_TYPE == "smolvla":
    MODEL_FLAGS = f"""\
    --policy.type=smolvla \\
    --policy.pretrained_path=lerobot/smolvla_base \\
    --policy.freeze_vision_encoder=false \\
    --policy.train_expert_only=false \\
    --policy.device=cuda"""

print(f"Model-specific flags:\n{MODEL_FLAGS}")

Model-specific flags:
    --policy.type=smolvla \
    --policy.pretrained_path=lerobot/smolvla_base \
    --policy.compile_model=true \
    --policy.gradient_checkpointing=true \
    --policy.freeze_vision_encoder=false \
    --policy.dtype=bfloat16 \
    --policy.device=cuda


In [10]:
# Train the policy. The trained model is pushed to the Hugging Face Hub under POLICY_REPO after training.
!accelerate launch \
--multi_gpu \
--num_machines=1 \
--num_processes=4 \
--mixed_precision=bf16 \
$(which lerobot-train) \
    --dataset.repo_id={DATASET_REPO} \
    --dataset.root={DATASET_ROOT} \
    --policy.push_to_hub=true \
    --policy.repo_id={POLICY_REPO} \
    --output_dir={OUTPUT_DIR} \
    --job_name={JOB_NAME} \
    --wandb.enable=true \
    --steps={MAX_TRAIN_STEPS} \
    --log_freq=50 \
    --eval_freq=-1 \
    --policy.chunk_size={CHUNK_SIZE} \
    --policy.n_action_steps={ACTION_STEPS} \
    --batch_size={BATCH_SIZE} \
    {MODEL_FLAGS}

Traceback (most recent call last):
  File "/home/gimarchetti_google_com/miniconda3/envs/mujoco2/lib/python3.12/site-packages/draccus/parsers/decoding.py", line 357, in _try_functions
    return func(val, path)
           ^^^^^^^^^^^^^^^
  File "/home/gimarchetti_google_com/miniconda3/envs/mujoco2/lib/python3.12/site-packages/draccus/parsers/decoding.py", line 201, in decode_choice_class
    return decode_dataclass(subcls, raw_value, path)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/gimarchetti_google_com/miniconda3/envs/mujoco2/lib/python3.12/site-packages/draccus/parsers/decoding.py", line 149, in decode_dataclass
    raise DecodingError(path, f"The fields {formatted_keys} are not valid for {stringify_type(cls)}")
draccus.utils.DecodingError: `policy`: The fields `compile_model`, `gradient_checkpointing`, `dtype` are not valid for SmolVLAConfig

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home